# Compare Models
This notebook scores one or more trained models against a shared, labelled dataset and produces:

- A confusion matrix for each model
- A combined ROC curve and a combined Precision-Recall curve, with one line per model, averaged across all evaluation classes
- Optionally, a ROC and Precision-Recall curve zoomed in on one specific class

Everything it needs comes from `Models/` and `Data/` — see the [README](../README.md) for how those folders are laid out.

**How to use this notebook:** edit only the single "Config" cell below, then run every cell top to bottom (Run All). Nothing past the Config section needs to change.

## Setup
This requires tensorflow, numpy, matplotlib, and scikit-learn. If you're running this on Anaconda's base environment (the setup this project expects), numpy, matplotlib, and scikit-learn are already installed. TensorFlow is the one exception — install it first with:

```
pip install tensorflow
```

In [ ]:
import os
import re
import json
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc, confusion_matrix

## Config — the only cell you should need to edit

| Setting | What it controls |
|---|---|
| `DATASET_DIR` | The folder of images to test the models on. Must contain one subfolder per class (e.g. `Carabidae/`, `Chrysomelidae/`), each full of that class's photos — no separate training/testing split needed here, since nothing gets trained. |
| `EVAL_CLASSES` | The list of class names everything gets scored against. This is the "ruler" every model is measured with. |
| `DATASET_LABEL_MAPPING` | Only needed if `DATASET_DIR`'s subfolders are more specific than `EVAL_CLASSES` (e.g. the folder has `Curculionidae/` but you want it counted as `Other`). Leave as `{}` if the folder names already match `EVAL_CLASSES` exactly. |
| `MODELS` | The models you want to compare. Add, remove, or rename entries freely — each one points at a `best_weights_ResNet50.h5` file inside `Models/`. |
| `TARGET_CLASS` | Set this to one class name (in quotes) to also get a zoomed-in ROC/PR plot for just that class, or leave as `None` to skip it. |
| `AVERAGING` | `"macro"` weights every class equally; `"micro"` weights every image equally. `"macro"` is the more common choice when class sizes are uneven. |

**A model whose classes don't exactly match `EVAL_CLASSES`** (for example, a 9-class model being scored against a 4-class scheme) can supply a `"mapping"` that folds its extra classes into one of `EVAL_CLASSES` — usually `"Other"`. See the commented-out `"Museum+Collection"` example below.

Predictions are cached under `Presentation_Plots/cache/`, so re-running with the same dataset and models re-uses the earlier results instead of re-scoring from scratch — changing anything in this cell automatically invalidates the relevant cache entries.

In [ ]:
BASE = r"C:\Users\timl9\Downloads\Specimen_Classifier"

# Dataset to evaluate on. Must be a flat folder of class-named subfolders
# (folder/ClassName/*.jpg), no train/test split.
DATASET_DIR = os.path.join(BASE, "Data", "Merged_Collection_Museum_Toy")

# Ground-truth class scheme everything gets scored against.
EVAL_CLASSES = ["Carabidae", "Chrysomelidae", "Other", "Staphylinidae"]

# Only needed if DATASET_DIR's subfolders are more fine-grained than EVAL_CLASSES,
# e.g. {"Curculionidae": "Other", "Scarabaeidae": "Other"}. Any subfolder name not
# listed here is assumed to already match an entry in EVAL_CLASSES exactly.
DATASET_LABEL_MAPPING = {}

# Models to compare -- add/remove/edit entries freely.
#   weights : path to the .h5 file
#   classes : the model's own output class order, as trained (Keras sorts subfolder
#             names alphabetically, so this is just the alphabetically-sorted list of
#             class folders the model was trained on)
#   mapping : only needed if `classes` isn't identical to EVAL_CLASSES -- maps each
#             extra model class to one of EVAL_CLASSES. Omit (or {}) if they match.
MODELS = {
    "InHouse_June": {
        "weights": os.path.join(BASE, "Models", "InHouse_0603_ResNet50_1", "best_weights_ResNet50.h5"),
        "classes": ["Carabidae", "Chrysomelidae",
                    "Other", "Staphylinidae"],
        "mapping": {}
    },
    "InHouse_July": {
        "weights": os.path.join(BASE, "Models", "InHouse_0621_ResNet50_1", "best_weights_ResNet50.h5"),
        "classes": ["Carabidae", "Chrysomelidae",
                    "Other", "Staphylinidae"],
        "mapping": {}
    }
}
    # Example of a model with extra classes collapsed into EVAL_CLASSES:
    # "Museum+Collection": {
    #     "weights": os.path.join(BASE, "Models", "Museum+Collection_ResNet50_1", "best_weights_ResNet50.h5"),
    #     "classes": ["Carabidae", "Cerambycidae", "Chrysomelidae", "Corylophidae", "Curculionidae",
    #                 "Other", "Ptilodactylidae", "Scarabaeidae", "Staphylinidae"],
    #     "mapping": {"Cerambycidae": "Other", "Corylophidae": "Other", "Curculionidae": "Other",
    #                 "Ptilodactylidae": "Other", "Scarabaeidae": "Other"},
    # },


TARGET_CLASS = None          # a class name for its own ROC/PR plot, or None to skip and only get the combined plots
AVERAGING = "macro"          # "macro" (equal weight per class) or "micro" (equal weight per image)

img_height, img_width = (224, 224)
batch_size = 128
OUTPUT_DIR = os.path.join(BASE, "Presentation_Plots")
CACHE_DIR = os.path.join(OUTPUT_DIR, "cache")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Will evaluate {list(MODELS.keys())} on {DATASET_DIR}")

---
## Everything below this line runs automatically — no need to edit anything past this point.

The next few cells define helper functions; the actual comparison runs in the cells under **Run it**, further down.

In [ ]:
def sanitize(s):
    return re.sub(r'[^A-Za-z0-9]+', '', s)


# Every output filename below is tagged with the dataset + models that produced it, so
# re-running with a different DATASET_DIR/MODELS config creates new files instead of
# silently overwriting a previous run's plots.
DATASET_TAG = sanitize(os.path.basename(DATASET_DIR.rstrip(os.sep)))
MODELS_TAG = "_".join(sorted(sanitize(l) for l in MODELS.keys()))
RUN_TAG = f"{DATASET_TAG}__{MODELS_TAG}"


def cache_key(*parts):
    return hashlib.md5("|".join(str(p) for p in parts).encode()).hexdigest()[:16]


def align_to_eval_classes(probs, model_classes, mapping, eval_classes):
    """Collapse a model's raw (n_samples, len(model_classes)) probabilities down to
    (n_samples, len(eval_classes)) by summing each model class's probability into
    whichever eval class it maps to (identity if not listed in `mapping`)."""
    aligned = np.zeros((probs.shape[0], len(eval_classes)))
    for j, mcls in enumerate(model_classes):
        target = mapping.get(mcls, mcls)
        if target not in eval_classes:
            raise ValueError(
                f"Model class '{mcls}' maps to '{target}', which isn't in EVAL_CLASSES {eval_classes}. "
                f"Add '{mcls}' to this model's 'mapping' in Config."
            )
        aligned[:, eval_classes.index(target)] += probs[:, j]
    return aligned

In [ ]:
def get_dataset_true_labels():
    """Scan DATASET_DIR once, return (generator, true_classes_in_eval_space)."""
    from tensorflow.keras.applications.resnet50 import preprocess_input
    from tensorflow.keras.preprocessing.image import ImageDataGenerator

    datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
    generator = datagen.flow_from_directory(
        DATASET_DIR,
        target_size=(img_height, img_width),
        batch_size=batch_size,
        shuffle=False,
        class_mode='categorical')

    raw_class_names = list(generator.class_indices.keys())
    for name in raw_class_names:
        target = DATASET_LABEL_MAPPING.get(name, name)
        if target not in EVAL_CLASSES:
            raise ValueError(
                f"Dataset subfolder '{name}' maps to '{target}', which isn't in EVAL_CLASSES {EVAL_CLASSES}. "
                f"Add '{name}' to DATASET_LABEL_MAPPING in Config."
            )
    raw_to_eval_idx = np.array([EVAL_CLASSES.index(DATASET_LABEL_MAPPING.get(n, n)) for n in raw_class_names])
    true_classes = raw_to_eval_idx[generator.classes]
    print(f"Dataset: {generator.samples} images, folders {raw_class_names} -> eval classes {EVAL_CLASSES}")
    return generator, true_classes


def get_model_predictions(label, cfg, generator):
    """Load cached aligned predictions for this model+dataset+eval-scheme combo, or
    compute (and cache) them fresh if nothing matches."""
    key = cache_key(label, cfg["weights"], DATASET_DIR, EVAL_CLASSES, sorted(cfg.get("mapping", {}).items()))
    cache_path = os.path.join(CACHE_DIR, f"{key}.npy")
    if os.path.exists(cache_path):
        print(f"[{label}] using cached predictions ({cache_path})")
        return np.load(cache_path)

    import tensorflow as tf
    print(f"[{label}] loading model from {cfg['weights']} ...")
    model = tf.keras.models.load_model(cfg["weights"])
    generator.reset()
    raw_preds = model.predict(generator, verbose=1)
    aligned = align_to_eval_classes(raw_preds, cfg["classes"], cfg.get("mapping", {}), EVAL_CLASSES)
    np.save(cache_path, aligned)
    return aligned

In [ ]:
def macro_average_roc(true_classes, results):
    """One (fpr, tpr) curve per model, averaged equally across EVAL_CLASSES."""
    curves = {}
    for label, probs in results.items():
        all_fpr = np.unique(np.concatenate([
            roc_curve((true_classes == ci).astype(int), probs[:, ci])[0] for ci in range(len(EVAL_CLASSES))
        ]))
        mean_tpr = np.zeros_like(all_fpr)
        for ci in range(len(EVAL_CLASSES)):
            fpr, tpr, _ = roc_curve((true_classes == ci).astype(int), probs[:, ci])
            mean_tpr += np.interp(all_fpr, fpr, tpr)
        mean_tpr /= len(EVAL_CLASSES)
        curves[label] = (all_fpr, mean_tpr, auc(all_fpr, mean_tpr))
    return curves


def micro_average_roc(true_classes, results):
    """One (fpr, tpr) curve per model, pooling every (image, class) decision equally."""
    n_classes = len(EVAL_CLASSES)
    one_hot = np.eye(n_classes)[true_classes]
    curves = {}
    for label, probs in results.items():
        fpr, tpr, _ = roc_curve(one_hot.ravel(), probs.ravel())
        curves[label] = (fpr, tpr, auc(fpr, tpr))
    return curves


def macro_average_pr(true_classes, results):
    curves = {}
    for label, probs in results.items():
        all_recall = np.unique(np.concatenate([
            precision_recall_curve((true_classes == ci).astype(int), probs[:, ci])[1] for ci in range(len(EVAL_CLASSES))
        ]))
        mean_precision = np.zeros_like(all_recall)
        for ci in range(len(EVAL_CLASSES)):
            precision, recall, _ = precision_recall_curve((true_classes == ci).astype(int), probs[:, ci])
            # sklearn returns recall in descending order; np.interp needs ascending x
            mean_precision += np.interp(all_recall, recall[::-1], precision[::-1])
        mean_precision /= len(EVAL_CLASSES)
        curves[label] = (all_recall, mean_precision, auc(all_recall, mean_precision))
    return curves


def micro_average_pr(true_classes, results):
    n_classes = len(EVAL_CLASSES)
    one_hot = np.eye(n_classes)[true_classes]
    curves = {}
    for label, probs in results.items():
        precision, recall, _ = precision_recall_curve(one_hot.ravel(), probs.ravel())
        curves[label] = (recall, precision, auc(recall, precision))
    return curves


def plot_confusion_matrix(cm, labels, title, out_path):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    print(f"Saved {out_path}")
    plt.show()

## Run it
Run each cell below in order. Every plot is shown here in the notebook *and* saved as a PNG into `Presentation_Plots/`.

In [ ]:
generator, true_classes = get_dataset_true_labels()
results = {label: get_model_predictions(label, cfg, generator) for label, cfg in MODELS.items()}

for label, probs in results.items():
    acc = np.mean(np.argmax(probs, axis=1) == true_classes)
    print(f"[{label}] accuracy: {acc:.4f}")

### Per-class ROC / Precision-Recall
Only runs if `TARGET_CLASS` was set to a class name in Config; skipped if it's `None`.

In [ ]:
if TARGET_CLASS is not None:
    class_index = EVAL_CLASSES.index(TARGET_CLASS)
    class_true_labels = (true_classes == class_index).astype(int)

    plt.figure(figsize=(6, 5))
    for label, probs in results.items():
        fpr, tpr, _ = roc_curve(class_true_labels, probs[:, class_index])
        plt.plot(fpr, tpr, label=f'{label} (AUC={auc(fpr, tpr):.2f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    plt.xlim([0, 1]); plt.ylim([0, 1.05])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'ROC - {TARGET_CLASS}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'ROC_{TARGET_CLASS}_{RUN_TAG}.png'), dpi=150)
    plt.show()

    plt.figure(figsize=(6, 5))
    for label, probs in results.items():
        precision, recall, _ = precision_recall_curve(class_true_labels, probs[:, class_index])
        plt.plot(recall, precision, label=f'{label} (AUC={auc(recall, precision):.2f})')
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.title(f'Precision-Recall - {TARGET_CLASS}')
    plt.legend(loc='lower left')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'PR_{TARGET_CLASS}_{RUN_TAG}.png'), dpi=150)
    plt.show()
else:
    print("TARGET_CLASS is None in Config -- skipping the per-class plot.")

### Combined ROC / Precision-Recall
One line per model, averaged across every class in `EVAL_CLASSES` (using whichever `AVERAGING` mode Config specifies).

In [ ]:
roc_fn = micro_average_roc if AVERAGING == "micro" else macro_average_roc
pr_fn = micro_average_pr if AVERAGING == "micro" else macro_average_pr

plt.figure(figsize=(6, 5))
for label, (x, y, area) in roc_fn(true_classes, results).items():
    plt.plot(x, y, label=f'{label} (AUC={area:.2f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
plt.xlim([0, 1]); plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'{AVERAGING.capitalize()}-averaged ROC (all {len(EVAL_CLASSES)} classes)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'ROC_{AVERAGING}_average_{RUN_TAG}.png'), dpi=150)
plt.show()

plt.figure(figsize=(6, 5))
for label, (x, y, area) in pr_fn(true_classes, results).items():
    plt.plot(x, y, label=f'{label} (AUC={area:.2f})')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title(f'{AVERAGING.capitalize()}-averaged Precision-Recall (all {len(EVAL_CLASSES)} classes)')
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'PR_{AVERAGING}_average_{RUN_TAG}.png'), dpi=150)
plt.show()

### Confusion matrix per model

In [ ]:
for label, probs in results.items():
    cm = confusion_matrix(true_classes, np.argmax(probs, axis=1), labels=range(len(EVAL_CLASSES)))
    safe_label = sanitize(label)
    plot_confusion_matrix(cm, EVAL_CLASSES, f'Confusion Matrix - {label}',
                           os.path.join(OUTPUT_DIR, f'ConfusionMatrix_{safe_label}_{DATASET_TAG}.png'))

print("\nDone. All figures saved to:", OUTPUT_DIR)